In [22]:
import torch
import torch.nn as nn

# 1. THE BUILDING BLOCK: The "Bottleneck"
# This is the 3-layer sandwich (1x1, 3x3, 1x1) that makes ResNet-50 efficient.
class Bottleneck(nn.Module):
    expansion = 4  # The output channels are always 4x the input of the block
    def __init__(self, in_channels, out_channels, stride=1, downsample=None):
        super(Bottleneck, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)

        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.conv3 = nn.Conv2d(out_channels, out_channels * self.expansion, kernel_size=1, bias=False)
        self.bn3 = nn.BatchNorm2d(out_channels * self.expansion)

        self.relu = nn.ReLU(inplace=True)
        self.downsample = downsample

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu(out)

        out = self.conv3(out)
        out = self.bn3(out)

        # THE SKIP CONNECTION:
        # If the shape changed, we 'downsample' the identity to match 'out'
        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity  # This is the core "ResNet" magic (Residual addition)
        out = self.relu(out)
        return out


class ResNet(nn.Module):
    def __init__(self, block, layers, num_classes=10):
        super(ResNet, self).__init__()
        self.in_channels = 64


        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)


        self.layer1 = self._make_layer(block, 64,  layers[0])
        self.layer2 = self._make_layer(block, 128, layers[1], stride=2)
        self.layer3 = self._make_layer(block, 256, layers[2], stride=2)
        self.layer4 = self._make_layer(block, 512, layers[3], stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512 * block.expansion, num_classes)

    def _make_layer(self, block, out_channels, blocks, stride=1):
        downsample = None

        if stride != 1 or self.in_channels != out_channels * block.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(self.in_channels, out_channels * block.expansion, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels * block.expansion),
            )

        layers = []
        layers.append(block(self.in_channels, out_channels, stride, downsample))
        self.in_channels = out_channels * block.expansion
        for _ in range(1, blocks):
            layers.append(block(self.in_channels, out_channels))

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x

def resnet50(num_classes=10):
    return ResNet(Bottleneck, [3, 4, 6, 3], num_classes)


device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

model = resnet50().to(device)

print(f"Model is running on: {device}")

total_params = sum(p.numel() for p in model.parameters())
print(f"Total Parameters: {total_params:,}")

Model is running on: mps
Total Parameters: 23,520,842


In [23]:
loss_fn = nn.CrossEntropyLoss()

optimiser = torch.optim.Adam(model.parameters(),lr=0.0001)

In [31]:
from torchvision import transforms
import torchvision

"""transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])
"""

transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5), # 50% chance to flip
    transforms.RandomRotation(15),          # Rotate by up to 15 degrees
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)), # Shift image slightly
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

In [33]:
shared_root='/Users/benjaminbrooke/PycharmProjects/Python_PyTroch/IT 599 Research Paper/1.LeNet-5/data/'

train_set = torchvision.datasets.CIFAR10(root=shared_root, train=True, download=True, transform=transform)

In [26]:
print(next(iter(train_set)))

(tensor([[[-0.5373, -0.6627, -0.6078,  ...,  0.2392,  0.1922,  0.1608],
         [-0.8745, -1.0000, -0.8588,  ..., -0.0353, -0.0667, -0.0431],
         [-0.8039, -0.8745, -0.6157,  ..., -0.0745, -0.0588, -0.1451],
         ...,
         [ 0.6314,  0.5765,  0.5529,  ...,  0.2549, -0.5608, -0.5843],
         [ 0.4118,  0.3569,  0.4588,  ...,  0.4431, -0.2392, -0.3490],
         [ 0.3882,  0.3176,  0.4039,  ...,  0.6941,  0.1843, -0.0353]],

        [[-0.5137, -0.6392, -0.6235,  ...,  0.0353, -0.0196, -0.0275],
         [-0.8431, -1.0000, -0.9373,  ..., -0.3098, -0.3490, -0.3176],
         [-0.8118, -0.9451, -0.7882,  ..., -0.3412, -0.3412, -0.4275],
         ...,
         [ 0.3333,  0.2000,  0.2627,  ...,  0.0431, -0.7569, -0.7333],
         [ 0.0902, -0.0353,  0.1294,  ...,  0.1608, -0.5137, -0.5843],
         [ 0.1294,  0.0118,  0.1137,  ...,  0.4431, -0.0745, -0.2784]],

        [[-0.5059, -0.6471, -0.6627,  ..., -0.1529, -0.2000, -0.1922],
         [-0.8431, -1.0000, -1.0000,  ..., -

In [27]:
from ptflops import get_model_complexity_info

test_model_parameter = resnet50(num_classes=10)

macs, params = get_model_complexity_info(test_model_parameter, (3, 32, 32), as_strings=True, print_per_layer_stat=True)

ResNet(
  23.52 M, 100.000% Params, 1.31 GMac, 99.773% MACs, 
  (conv1): Conv2d(1.73 k, 0.007% Params, 1.77 MMac, 0.135% MACs, 3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(128, 0.001% Params, 131.07 KMac, 0.010% MACs, 64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(0, 0.000% Params, 65.54 KMac, 0.005% MACs, inplace=True)
  (layer1): Sequential(
    215.81 k, 0.918% Params, 222.17 MMac, 16.951% MACs, 
    (0): Bottleneck(
      75.01 k, 0.319% Params, 77.2 MMac, 5.890% MACs, 
      (conv1): Conv2d(4.1 k, 0.017% Params, 4.19 MMac, 0.320% MACs, 64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(128, 0.001% Params, 131.07 KMac, 0.010% MACs, 64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(36.86 k, 0.157% Params, 37.75 MMac, 2.880% MACs, 64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(128, 0.001

In [34]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_set,shuffle= True, batch_size=30)

In [35]:
from tqdm.notebook import tqdm
import time
from torch.utils.tensorboard import SummaryWriter
LeNet5_Metrics = SummaryWriter()

latency_per_image = []
global_i = 0

pbar = tqdm(train_loader)

total=len(train_loader)

all_preds = []
all_labels = []

for image,label in pbar:

    image, label = image.to(device), label.to(device)

    start_time = time.time()

    optimiser.zero_grad()

    y_pred =  model(image)

    loss = loss_fn(y_pred,label)

    loss.backward()

    optimiser.step()

    _, predicted = torch.max(y_pred.data, 1)
    correct = (predicted == label).sum().item()
    accuracy = correct / label.size(0)

    end_time = time.time()

    latency_per_image.append(end_time - start_time)

    LeNet5_Metrics.add_scalar("Loss/train - ResNet-50 IT599 Ben", loss.item(), global_i)
    LeNet5_Metrics.add_scalar("Accuracy/train - ResNet-50 IT599 Ben", accuracy, global_i)

    all_preds.extend(predicted.cpu())
    all_labels.extend(label.cpu())

    global_i += 1

    print(global_i/total)


LeNet5_Metrics.close()

avg_latency_batch = sum(latency_per_image) / len(latency_per_image)
avg_latency_image = avg_latency_batch / train_loader.batch_size
throughput = 1 / avg_latency_image

print(f"Latency: {avg_latency_image*1000:.2f} ms | Throughput: {throughput:.2f} items/sec")

  0%|          | 0/1667 [00:00<?, ?it/s]

0.0005998800239952009
0.0011997600479904018
0.001799640071985603
0.0023995200959808036
0.002999400119976005
0.003599280143971206
0.004199160167966407
0.004799040191961607
0.005398920215956809
0.00599880023995201
0.0065986802639472104
0.007198560287942412
0.007798440311937612
0.008398320335932814
0.008998200359928014
0.009598080383923215
0.010197960407918417
0.010797840431913617
0.011397720455908818
0.01199760047990402
0.01259748050389922
0.013197360527894421
0.013797240551889621
0.014397120575884824
0.014997000599880024
0.015596880623875225
0.016196760647870425
0.016796640671865627
0.01739652069586083
0.017996400719856028
0.01859628074385123
0.01919616076784643
0.01979604079184163
0.020395920815836834
0.020995800839832032
0.021595680863827234
0.022195560887822437
0.022795440911817635
0.023395320935812838
0.02399520095980804
0.02459508098380324
0.02519496100779844
0.02579484103179364
0.026394721055788842
0.026994601079784044
0.027594481103779243
0.028194361127774445
0.028794241151769647

In [36]:
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(all_labels, all_preds))

              precision    recall  f1-score   support

           0       0.67      0.68      0.68      5000
           1       0.78      0.79      0.79      5000
           2       0.54      0.52      0.53      5000
           3       0.46      0.44      0.45      5000
           4       0.58      0.57      0.57      5000
           5       0.55      0.54      0.54      5000
           6       0.71      0.74      0.72      5000
           7       0.69      0.69      0.69      5000
           8       0.75      0.77      0.76      5000
           9       0.73      0.75      0.74      5000

    accuracy                           0.65     50000
   macro avg       0.65      0.65      0.65     50000
weighted avg       0.65      0.65      0.65     50000



In [ ]:
#python3 -m tensorboard.main --logdir="/Users/benjaminbrooke/PycharmProjects/Python_PyTroch/IT 599 Research Paper/4.ResNet-50/runs"

In [37]:
torch.save(model.state_dict(),"/Users/benjaminbrooke/PycharmProjects/Python_PyTroch/IT 599 Research Paper/4.ResNet-50/ResNet50")